# Lab 04.1 — OpenCV Image Basics

## Objectives

- verify that OpenCV is available;
- create and save a controlled test image;
- understand image dimensions and pixel values;
- compare BGR, RGB and grayscale representations;
- select a region of interest (ROI).

This first exercise uses a static image. From Lab 04.2 onward we switch to the Logitech C270 webcam.

Keep this introduction short (about 10 minutes). Run each notebook from top to bottom in its own Python 3 kernel on Zybo. OpenCV 4.5.4 is the version already tested on this board; no package upgrade is required.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("OpenCV version:", cv2.__version__)

## 1. Create a reproducible test image

OpenCV stores color images in **BGR** channel order. We generate a synthetic scene with strongly colored objects so that later segmentation experiments are predictable.

In [ ]:
Path("images").mkdir(exist_ok=True)

BACKGROUND = 110  # Mid-gray: all four shapes remain visible in grayscale.
img = np.full((360, 640, 3), BACKGROUND, dtype=np.uint8)

cv2.rectangle(img, (40, 60), (180, 200), (255, 0, 0), -1)      # blue in BGR
cv2.circle(img, (320, 130), 75, (0, 255, 0), -1)               # green
triangle = np.array([[470, 210], [560, 50], [625, 210]])
cv2.fillPoly(img, [triangle], (0, 0, 255))                      # red
cv2.rectangle(img, (220, 250), (420, 330), (0, 255, 255), -1)  # yellow

if not cv2.imwrite("images/colored_shapes.png", img):
    raise IOError("Could not save images/colored_shapes.png; check folder permissions.")

print("Shape:", img.shape)
print("Data type:", img.dtype)

In [ ]:
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 5))
plt.imshow(img_rgb)
plt.title("Test image")
plt.axis("off");

## 2. Read the image from file

`cv2.imread()` returns a NumPy array. For a color image, `shape` is:

`(height, width, channels)`

In [ ]:
image = cv2.imread("images/colored_shapes.png")
if image is None:
    raise FileNotFoundError("Run the image-generation cell above first.")

print("Height:", image.shape[0])
print("Width :", image.shape[1])
print("Channels:", image.shape[2])

print("Pixel at (100, 100), BGR =", image[100, 100])

## 3. BGR, RGB and grayscale

OpenCV normally uses BGR, while Matplotlib expects RGB. Grayscale replaces the three color channels with one intensity value; it does not remove yellow objects.

OpenCV uses approximately:

`gray = 0.299 * R + 0.587 * G + 0.114 * B`

Pure yellow has R=255, G=255 and B=0, so it becomes about **226** on the 0–255 grayscale. On the earlier light-gray background (235), the rectangle was still present but almost blended into the background: their intensities differed by only 9. The revised image uses background 110 so all four shapes remain visible.

We fix the display scale with `vmin=0, vmax=255`: 0 is black and 255 is white. This avoids automatic rescaling; it does not restore color information lost in conversion.

[OpenCV 4.5.4: color conversions](https://docs.opencv.org/4.5.4/de/d25/imgproc_color_conversions.html)

In [ ]:
rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(rgb)
axes[0].set_title("RGB")
axes[1].imshow(gray, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Grayscale: yellow is light gray")
for ax in axes:
    ax.axis("off")
plt.show()
plt.close(fig)

# Known interior points from the scene generated above:
print("Yellow rectangle intensity:", int(gray[290, 320]))  # approximately 226
print("Background intensity:", int(gray[10, 10]))           # 110


## 4. Region of interest (ROI)

A ROI is a rectangular part of an image. **First inspect the complete image with the coordinate axes and grid below; you do not need to guess or memorize the shapes' coordinates.**

The origin (0, 0) is at the top left. **x** is the column and increases to the right; **y** is the row and increases downward. Read x on the horizontal axis and y on the vertical axis.

NumPy indexes rows before columns:

`roi = rgb[top:bottom, left:right]` = `rgb[y_start:y_stop, x_start:x_stop]`

The start is included and the stop is excluded. All three RGB channels are kept. A small background margin around the selected shape is fine.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(rgb)
ax.set_title("Original image — use this grid to choose coordinates")
ax.set_xlabel("x — column (left to right)")
ax.set_ylabel("y — row (top to bottom)")
ax.set_xticks(np.arange(0, rgb.shape[1], 50))
ax.set_yticks(np.arange(0, rgb.shape[0], 50))
ax.grid(True, color="black", alpha=0.35)
plt.show()
plt.close(fig)


### Worked example: the blue square

From the grid, the blue square extends approximately from x=40 to x=180 and y=60 to y=200. Choose a small margin: left=20, right=200, top=40, bottom=220.

This selects rows 40…219 and columns 20…199, producing a 180×180-pixel image. Cropping keeps background inside the rectangle; selecting only the colored pixels will require a mask, introduced in Lab04.3.

In [ ]:
top, bottom = 40, 220
left, right = 20, 200

roi = rgb[top:bottom, left:right]
print("ROI shape:", roi.shape)

plt.figure(figsize=(5, 4))
plt.imshow(roi)
plt.title("Blue square ROI")
plt.axis("off")
plt.show()


## Short exercise

1. **Crop the green circle.** Use the original image with the grid. Estimate the left, right, top and bottom edges of the circle, then change the four variables in the ROI cell above. Run that cell again. Keep the whole circle; a small background margin is acceptable. If part is cut off, enlarge the selected region. Exact coordinates are not required.
2. **Inspect a red pixel.** On the same grid, choose a point well inside the red triangle. Read its x and y coordinates and use `image[y, x]` for BGR or `rgb[y, x]` for RGB. A worked example is supplied below; then try another nearby interior point.
3. Explain why the same red pixel has different channel order in `image` and `rgb`.

Use the **original image's coordinates**, not the coordinates of the cropped ROI, for the pixel exercise.

In [ ]:
# The grid shows that x=550, y=150 is inside the red triangle.
x, y = 550, 150
print("BGR:", image[y, x])  # expected [0, 0, 255]
print("RGB:", rgb[y, x])    # expected [255, 0, 0]
